# 01 C++ 核心语法与 STL

> 覆盖范围：值类别（左值/右值与移动语义）、指针/引用/const、内存模型（栈/堆/RAII/智能指针）、类与多态（虚函数/vtable/虚析构）、模板（实例化/特化/SFINAE）与 STL（迭代器类别/容器复杂度/算法）、现代 C++（11/14/17/20）高频特性。
> 使用方式：每节先读**原理讲解**（为什么存在 + 底层机制），再看 **C++ 代码示例**（用于阅读与默写），最后用 Python 对照实验建立直觉，自测面试追问。
> 注：本 notebook 内核为 Python 3；C++ 片段用于阅读与默写，对照实验用 Python 模拟以保持可执行验证。

---

## 1. 值类别：左值 vs 右值（C++11 的核心革命）

### 1.1 什么是左值 / 右值

C++11 把表达式按「身份与可移动性」分为几种类别，面试只需抓住**两种**：

- **左值（lvalue）**：**有名字、可取地址**、生命周期延续到表达式之外（变量、`*p`、数组元素、返回引用的函数调用）；
- **右值（rvalue）**：**临时值/即将销毁的值**（字面量 `42`、表达式 `a+b`、返回值的函数调用），没有持久身份。

```cpp
int a = 42;        // a 是左值；42 是右值
int b = a + 1;     // a+1 是右值（临时）
int& r = a;        // 左值引用绑定左值
int&& rr = 42;     // 右值引用绑定右值（也能绑定 std::move(a) 的结果）
std::move(a);      // 把左值“伪装”成右值，便于被“移动”
```

### 1.2 为什么要区分：移动语义

问题是：**拷贝很贵，临时对象马上要销毁，何必深拷贝？**

- 旧 C++：`std::vector<T> v2 = v1;` 一定是深拷贝（即使 v1 是临时）；
- C++11 起：**右值引用 `&&` + 移动构造**——识别「即将销毁的右值」，把它的内部资源（如堆指针）**直接转移**给新对象，旧对象指针置空。

```cpp
// 移动构造示例（String 类）
String(String&& o) noexcept
    : buf(o.buf), len(o.len)      // 直接“偷”资源
{ o.buf = nullptr; o.len = 0; }  // 旧对象置空，析构安全
```

- `std::move(x)` **不是搬数据**，只是一个 `static_cast` 把左值转成右值引用，让编译器「选择移动而不是拷贝」；
- **完美转发** `template<typename T> void f(T&& t)` + `std::forward<T>(t)`：保留实参的左右值属性传给下游（`T&&` 是转发引用，配合引用折叠）。

> **一句话**：右值引用 + 移动构造 = 把「即将销毁的临时资源」直接转交，避免深拷贝；`std::move` 只是静态类型转换，不搬任何东西。

## 2. 指针 vs 引用

| 维度 | 指针 `T*` | 引用 `T&` |
| --- | --- | --- |
| 初始化 | 可延后；可为 `nullptr` | **必须初始化**；不可为空 |
| 重绑 | 可换个对象指 | 一旦绑定，不可改绑 |
| 语法 | `*p` / `p->m` 解引用 | 编译器帮你解引用，`r` 直接用 |
| 额外操作 | 指针运算 `p+i`、地址比较 | 无（引用不是对象，无独立地址） |
| 典型用途 | 动态分配、容器元素、手动生命周期 | 函数入参避免拷贝、运算符重载、别名 |

```cpp
void swap(int& a, int& b) { int t = a; a = b; b = t; }          // 引用传参
void f(const std::string& s) { /* 只读重用，不拷贝 */ }
int* p = new int(3); delete p;                                   // 裸指针需手动管理
int arr[3] = {1,2,3}; int* q = arr;   // 数组名退化为指针
```

**const 的三种位置**：`const int* p`（指向的 int 不可改，指针可移）、`int* const p`（指针本身不可改）、`const int* const p`（都不可改）。

**传参经验**：

- 小对象（int/double/小 struct）**按值**；
- 大对象只读 **`const T&`**（避免拷贝）；
- 需要改入参 **`T&`**；
- 想要「可能没有」/可重绑/放进容器用**指针**。

## 3. 内存模型：栈 / 堆 / 生命周期

- **栈（stack）**：函数局部对象，**自动分配/释放**（出作用域析构），快、大小受限（默认几 MB），**LIFO**；
- **堆（heap）**：`new`/`delete`（或 `malloc`/`free`）显式管理，慢、任意大小，**需要手动释放**；
- **静态/全局存储区**：程序生命周期内存在；**代码段/常量区**、**线程局部存储**等。

**三类典型 bug**：

- **内存泄漏**：new 了忘了 delete（密集循环里会 OOM）；
- **悬垂指针（dangling）**：对象已析构（栈返回局部对象地址、delete 后仍用指针），访问是未定义行为；
- **双重释放 / 野指针**：delete 两次 / 未初始化指针。

**解决方案就是 RAII**（见下节）——把「手动管理」变成「生命周期管理」。

## 4. RAII 与智能指针

### 4.1 RAII 是什么

**RAII（Resource Acquisition Is Initialization）**：资源在**构造函数中获取，析构函数中释放**。由于 C++ 保证局部对象**离开作用域必然析构**（即使中途 return / 抛异常——栈展开 stack unwinding），资源释放就「必然发生」，天然异常安全。

```cpp
// 打开文件 -> 离开作用域自动关闭（析构）
{ std::ofstream out("a.txt"); out << "hi"; }  // 这里自动 close
```

适用资源：堆内存、文件、锁（`std::lock_guard`）、套接字、数据库连接、GPU 缓冲。

### 4.2 三种智能指针

| 指针 | 语义 | 用途 |
| --- | --- | --- |
| `std::unique_ptr<T>` | **独占**所有权，不可拷贝、可移动 | 默认首选；工厂函数返回；`make_unique` |
| `std::shared_ptr<T>` | **共享**所有权，引用计数 | 多个所有者共享资源；`make_shared` |
| `std::weak_ptr<T>` | **弱引用**，不增加计数 | 打破 shared_ptr **循环引用**；缓存观察者 |

```cpp
// shared_ptr 计数语义
auto s1 = std::make_shared<int>(42);   // use_count = 1
{ auto s2 = s1; }                      // 拷贝 -> +1，作用域结束 -> -1
// 出作用域后 use_count 归 0 自动 delete

// 循环引用：A 持有 B，B 持有 A（shared_ptr 互相持有）
// -> 计数永不归零 -> 泄漏！用 weak_ptr 打破（弱引用不计数）
```

> **面试要点**：`shared_ptr` 的引用计数本身是**线程安全**的（原子操作），但**被指向对象的读写不保证线程安全**；`enable_shared_from_this` 用于在成员函数里安全取回 `shared_ptr`。


## 核心概念

### (a) 🗂️ C++ 的编译、内存与多态主线

这篇把 C++ 最「硬核」的部分串成一条线：值类别与移动语义、指针与引用、内存模型与 RAII、智能指针，再到虚函数多态与模板两种多态，最后落到 STL 容器算法与现代 C++ 新特性——全是面试和实战的高频考点。

```mermaid
flowchart TB
    Node1["🧠 编译型语言：源码 → 编译 → 运行"] --> Node2["🏷️ 值类别：左值 vs 右值"]
    Node2 --> Node3["📦 指针 vs 引用"]
    Node1 --> Node4["🗄️ 内存模型：栈 / 堆 / RAII"]
    Node4 --> Node5["🧲 智能指针：unique / shared / weak"]
    Node3 --> Node6["🧬 类与多态：虚函数 / vtable"]
    Node6 --> Node7["🧩 模板：编译期多态"]
    Node7 --> Node8["🗃️ STL：容器 / 迭代器 / 算法"]
    Node8 --> Node9["✨ 现代 C++：11 / 14 / 17 / 20"]
```

- **🧠 编译型语言**：C++ 源码先编译成机器码再运行，没有解释器兜底——类型、内存、生命周期都要程序员心里有数。所以这篇讲的所有机制，本质都是「帮编译器管好资源」。
- **🏷️ 值类别：左值 vs 右值**：左值有名字、可取地址（变量），右值是临时的（`42`、`a+b`）。C++11 的 `&&` 专门识别右值，触发移动构造把「即将销毁的临时资源」直接转交，避免深拷贝；`std::move` 只是类型转换，不搬任何数据。
- **📦 指针 vs 引用**：指针可空、可重绑、要手动解引用；引用必须初始化、绑定后不可改、编译器替你解引用。传参经验：小对象按值、大对象只读用 `const T&`、要修改用 `T&`、允许为空或放进容器用指针。
- **🗄️ 内存模型：栈 / 堆 / RAII**：栈自动分配释放（出作用域析构）、快但小；堆 `new`/`delete` 手动管理、慢但大。三大经典 bug：内存泄漏、悬垂指针、双重释放。RAII 把资源获取放构造、释放放析构，作用域结束必然释放，天然异常安全。
- **🧲 智能指针**：`unique_ptr` 独占所有权、可移动不可拷贝，默认首选；`shared_ptr` 引用计数共享所有权；`weak_ptr` 不增加计数，专门打破 shared_ptr 循环引用（A 持 B、B 持 A 时计数永不归零）。
- **🧬 类与多态：虚函数 / vtable**：`virtual` 函数按对象实际类型动态绑定，机制是每个类一张 vtable、每个对象一个 vptr，调用走 `vptr -> vtable[i]`。基类析构必须 `virtual`，否则用基类指针 delete 派生对象会漏释放派生类资源。
- **🧩 模板：编译期多态**：模板按用到的类型在编译期实例化生成代码，无运行开销；特化为特定类型定制实现，SFINAE 把「替换失败」静默排除（`enable_if` 的底层），C++20 的 Concept 更优雅。它和虚函数是两种多态：编译期 vs 运行时。
- **🗃️ STL：容器 / 迭代器 / 算法**：选容器就是选底层结构——`vector` 连续数组默认首选、`list` 双向链表、`map/set` 红黑树有序 O(log n)、`unordered_map` 哈希平均 O(1)。迭代器分五级能力递增，算法会按迭代器能力自动降级。
- **✨ 现代 C++**：C++11 带来 `auto`/智能指针/移动语义/lambda/`nullptr`；14 出泛型 lambda；17 出 `if constexpr`/结构化绑定/`optional`；20 出 Concept/协程/三路比较 `<=>`。面试默写大多围绕这些新特性。


## 5. 类与多态：虚函数、vtable、虚析构

### 5.1 虚函数与动态绑定

- `virtual` 成员函数：调用时**按对象实际类型**分派（动态绑定），而不是按指针/引用的静态类型；
- **机制**：每个含虚函数的类，编译器生成一张 **vtable（虚函数表）**（存放各虚函数地址）；每个对象内部藏一个 **vptr（虚表指针）** 指向该类 vtable；调用虚函数 = 通过 `vptr -> vtable[i]` 间接跳转；
- 代价：每个对象多一个 vptr（8 字节）+ 一次间接跳转（比直接调用略慢）。

```cpp
struct Animal {
    virtual ~Animal() {}                        // 虚析构：必须！
    virtual std::string speak() const { return "?"; }
};
struct Dog : Animal {
    std::string speak() const override { return "woof"; }  // override 编译期检查
};
// Animal* a = new Dog();  a->speak() -> "woof" （通过 vtable 动态绑定）
```

### 5.2 为什么析构函数要 virtual

若用**基类指针** `delete` 派生类对象：

- 非虚析构：只调用**基类析构**（编译器按静态类型）→ 派生类资源（堆成员）**泄漏**；
- 虚析构：先调用派生析构（vptr 找到派生类）→ 再调用基类析构，**完整释放**。

### 5.3 纯虚函数与抽象类

`virtual void f() = 0;`——纯虚函数：类变成**抽象类**（不能实例化），派生类必须实现；用于定义接口。加 `final` 可禁止继续继承/覆写。

### 5.4 RTTI

运行时类型识别：`dynamic_cast<T*>(p)`（沿继承层次安全的向下转换，失败返回 nullptr）、`typeid(x).name()`。需要 vtable 支持，仅对多态类型有效。

## 6. 模板：编译期多态

### 6.1 函数模板 / 类模板

```cpp
template <typename T>
T max_of(T a, T b) { return a > b ? a : b; }   // 调用时按实参推导并实例化
// max_of(3, 5) -> 实例化 max_of<int>；max_of(1.2, 3.4) -> max_of<double>
```

- **实例化时机**：编译期按「用到的类型」生成代码（所以模板实现通常在头文件）；
- **特化**：`template<> struct Hash<MyType> {...}` 为特定类型定制实现；
- **SFINAE**：替换失败不是错误——模板重载选择时，失败的候选被「静默排除」（enable_if 的底层）；C++20 的 **Concept** 是更优雅的约束：`requires std::integral<T>`；
- **模板 vs 虚函数**：模板是编译期多态（无运行开销、类型安全、代码膨胀），虚函数是运行时多态（灵活、可扩展、有间接跳转开销）。

### 6.2 lambda 表达式（C++11）

```cpp
auto f = [cap](Params) -> Ret { body };  // cap: 捕获列表
// [=] 值捕获、[&] 引用捕获、[x, &y] 混合；[this] 捕获 this
// capture 默认为 const；mutable 允许在 lambda 内修改值捕获副本
std::sort(v.begin(), v.end(), [](int a, int b){ return a > b; });
```

**捕获的坑**：按引用捕获 `[&]` 后 lambda 生命周期超过变量 → 悬垂引用；默认捕获尽量用 `[=]` + 显式例外。

## 7. STL：容器 / 迭代器 / 算法

### 7.1 容器选型（复杂度表）

| 容器 | 底层 | 随机访问 | 插入/删除 | 查找 | 特点 |
| --- | --- | --- | --- | --- | --- |
| `std::vector` | 连续数组 | O(1) | 尾 O(1)*，中间 O(n) | 无序 O(n) | **默认首选**，缓存友好 |
| `std::list` | 双向链表 | O(n) | 已知位置 O(1) | O(n) | 中间频繁插入删除 |
| `std::deque` | 分段连续 | O(1) | 两端 O(1) | O(n) | 双端队列 |
| `std::set`/`map` | 红黑树 | O(n)/O(log n) | O(log n) | O(log n) | 有序 |
| `std::unordered_set/map` | 哈希表 | — | 平均 O(1) | 平均 O(1) | 无序、最快查重 |
| `std::priority_queue` | 堆 | — | push/pop O(log n) | — | 取极值 |

> `*`：vector `push_back` **均摊 O(1)**——满了翻倍扩容（2 倍），把 N 次 push 的总成本摊到每次：均摊 O(1)。

### 7.2 迭代器类别（五级）

`输入/输出迭代器 → 前向 → 双向 → 随机访问`：能力递增。`vector` 是随机访问迭代器；`list` 是双向；`unordered_map` 是前向。算法会**按迭代器能力降级**（例如 `std::distance` 对随机访问 O(1)、对双向 O(n)）。

### 7.3 常用算法（`<algorithm>`）

```cpp
std::sort / std::stable_sort / std::nth_element   // 排序/稳定排序/第k大
std::binary_search / std::lower_bound             // 有序二分
std::find / std::count / std::accumulate          // 查找/计数/累加
std::transform / std::copy / std::remove_if       // 转换/复制/删除（remove 后要 erase）
std::reverse / std::unique / std::minmax_element  // 反转/去重/极值
```

## 8. 现代 C++ 高频特性一览（11/14/17/20）

- **C++11**：`auto`、范围 for `for(auto& x : v)`、智能指针、右值引用/移动语义、lambda、`nullptr`、`override`/`final`、`constexpr`、`std::thread`；
- **C++14**：泛型 lambda（`auto` 参数）、`make_unique`、返回类型推导；
- **C++17**：`if constexpr`、**结构化绑定** `auto [a,b] = pair`、`std::optional`/`std::variant`/`std::string_view`、内联变量；
- **C++20**：**Concept**（`requires`）、协程、`std::span`、三路比较 `<=>`、范围库 `std::ranges`。

```cpp
// 结构化绑定 + if constexpr（C++17）
auto [k, v] = std::make_pair(1, std::string("a"));
template <typename T> void print(const T& x) {
    if constexpr (std::is_integral_v<T>) { /* 整型分支 */ }
    else { /* 其他 */ }
}
```


## 9. 对照实验（Python 模拟 C++ 核心语义）

下面用 Python 演示四个核心概念，帮助建立直觉：

1. **引用 vs 值**：Python 的对象绑定 ≈ C++ 的引用（对可变对象原地修改可见；不可变对象 ≈ 按值传递）；
2. **RAII 生命周期**：用 `with`/析构模拟作用域结束自动释放；
3. **多态**：Python 鸭子类型 ≈ C++ 虚函数动态绑定；
4. **shared_ptr 引用计数**：模拟拷贝增计数、归零释放。


In [ ]:
# ---- matplotlib 中文字体（防图内中文乱码）----
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
# -------------------------------------------
# 1) 引用语义对照（C++ 引用传参 ≈ Python 可变对象共享）
def swap_cpp_style(a):          # 模拟 void swap(int& a, int& b)
    a[0], a[1] = a[1], a[0]

arr = [1, 2]
swap_cpp_style(arr)
print('引用传参后 arr =', arr, ' <- 原地修改，类似 C++ 引用')

# 不可变对象不会被“引用修改”（类似按值传递 int）
x = 10
def try_modify(v):
    v += 1          # 局部绑定新对象
try_modify(x)
print('传 int 后 x =', x, ' <- 未变，类似按值拷贝')

In [ ]:
# 2) RAII 生命周期对照：作用域结束自动释放
class Resource:
    def __init__(self, name):
        self.name = name
        print(f'[构造] {name} 获取资源')
    def __del__(self):
        print(f'[析构] {self.name} 释放资源')

def scope_demo():
    r = Resource('unique_ptr 管理的堆对象')
    print('作用域内使用中...')
    return

scope_demo()
print('函数返回后，资源已自动释放（RAII）')

# 即使“异常”也释放：模拟栈展开
def scope_with_error():
    r = Resource('异常路径中的对象')
    raise ValueError('中途抛异常')

try:
    scope_with_error()
except ValueError as e:
    print('捕获异常:', e, '<- 析构仍执行了：RAII 异常安全')

In [ ]:
# 3) 多态对照：C++ 虚函数动态绑定 ≈ Python 鸭子类型
class Animal:
    def speak(self):
        return '?'

class Dog(Animal):
    def speak(self):      # 相当于 override
        return 'woof'

class Cat(Animal):
    def speak(self):
        return 'meow'

def zoo(animals):
    return [a.speak() for a in animals]   # 动态绑定：调用实际类型的方法

print('多态结果:', zoo([Dog(), Cat(), Animal()]))
print('<- C++ 中对应：Animal* 数组上调用虚函数，按对象实际类型分派')

# 4) shared_ptr 引用计数模拟
class ShPtr:
    def __init__(self, name):
        self.name = name
        self.refs = 1
    def copy(self):
        self.refs += 1
        return self      # 模拟 shared_ptr 拷贝：计数 +1，指向同一资源
    def destroy(self):
        self.refs -= 1
        if self.refs == 0:
            print(f'shared_ptr 引用计数归零 -> 释放 {self.name}')
            return True
        print(f'shared_ptr 剩余引用 {self.refs}')
        return False

s = ShPtr('相机内存')
s2 = s.copy()          # 第二个所有者
s3 = s.copy()          # 第三个所有者
s.destroy(); s2.destroy(); s3.destroy()   # 依次析构，最后归零才释放

In [ ]:
# 5) 容器复杂度对照表验证（Python 容器 vs C++ STL 概念）
# C++ 中 vector 随机访问 O(1)，list 中间插入 O(1)，unordered_map 查找平均 O(1)
# Python 中 list 底层是动态数组（类似 vector）、dict 是哈希表（类似 unordered_map）

import time

# dict 查重 O(1) 与 list 线性查 O(n) 对比（类似 unordered_map vs 手写线性查找）
n = 3000
d = {i: i for i in range(n)}
lst = list(range(n))

t0 = time.perf_counter()
r1 = sum(1 for i in range(n) if i in d)
t1 = time.perf_counter()
t2 = time.perf_counter()
r2 = sum(1 for i in range(n) if i in lst)
t3 = time.perf_counter()
print(f'dict 查重(哈希 O(1)): {t1-t0:.4f}s 命中{r1}')
print(f'list 查重(线性 O(n)): {t3-t2:.4f}s 命中{r2}')
print(f'耗时比 ~{max((t3-t2)/(t1-t0+1e-12), 1):.0f}x (n 越大差距越明显)')
print('<- 对应 C++: unordered_map.find() 平均 O(1) vs 手写循环线性查找')

# vector 均摊 O(1) 扩容模拟：翻倍扩容，总成本均摊
cap, size, copies = 0, 0, 0
for i in range(1000):
    if size == cap:
        new_cap = cap * 2 if cap else 1
        copies += size          # 迁移旧元素成本
        cap = new_cap
    size += 1
print(f'1000 次 push_back：最终容量 {cap}，总拷贝成本 {copies}（< 2N，均摊 O(1)）')

## 10. 手写 C++ 模板（面试默写）

```cpp
// 1) RAII 智能指针骨架（unique_ptr 简化版）
template <typename T>
class UniquePtr {
    T* p;
public:
    explicit UniquePtr(T* t = nullptr) : p(t) {}
    ~UniquePtr() { delete p; }                            // RAII 释放
    UniquePtr(const UniquePtr&) = delete;                 // 禁拷贝
    UniquePtr& operator=(const UniquePtr&) = delete;
    UniquePtr(UniquePtr&& o) noexcept : p(o.p) { o.p = nullptr; }   // 移动语义
    T& operator*() const { return *p; }
    T* get() const { return p; }
};

// 2) 手写 vector push_back 骨架（翻倍扩容 + 均摊）
template <typename T>
void push_back(T*& buf, int& size, int& cap, const T& v) {
    if (size == cap) {
        cap = cap ? cap * 2 : 1;                         // 扩容翻倍
        T* nb = new T[cap];
        for (int i = 0; i < size; ++i) nb[i] = buf[i];   // 迁移旧元素
        delete[] buf;
        buf = nb;
    }
    buf[size++] = v;
}

// 3) String 深拷贝/移动（面试高频）
// String(const String& o) { len = o.len; buf = new char[len+1]; strcpy(buf, o.buf); }
// String(String&& o) noexcept : buf(o.buf), len(o.len) { o.buf = nullptr; }

// 4) shared_ptr 引用计数骨架
// shared_ptr: 指向管理块 {T* ptr; long* count;}，拷贝 count++, 析构 count--, 归零 delete
```

## 11. 自测清单（面试追问）

- [ ] 左值与右值区别？`std::move` 到底做了什么？（只是类型转换，不搬数据）
- [ ] 移动构造 vs 拷贝构造？什么时候自动走移动？（临时对象/`std::move` 的右值）
- [ ] 指针与引用区别？为什么函数参数用 `const T&` 传大对象？
- [ ] `const int* p` / `int* const p` / `const int* const p` 各是什么？
- [ ] RAII 是什么？为什么异常安全？（局部对象必然析构 + 栈展开）
- [ ] `unique_ptr` / `shared_ptr` / `weak_ptr` 各解决什么问题？循环引用怎么破？
- [ ] 为什么基类析构要 `virtual`？vtable 存在哪？多态调用开销多大？
- [ ] 纯虚函数 / 抽象类 / `final` 的作用？
- [ ] 模板何时实例化？函数模板怎么推导类型？特化与偏特化区别？
- [ ] 模板与虚函数（编译期多态 vs 运行时多态）怎么选？
- [ ] `vector` 扩容机制？push_back 均摊复杂度为什么是 O(1)？（翻倍扩容，均摊 2 次拷贝/元素）
- [ ] `map` vs `unordered_map` 分别什么时候用？迭代器五类是什么？
- [ ] lambda 捕获 `[=]` / `[&]` / `[this]` 的区别？按引用捕获可能有什么坑？
- [ ] 手写：`strcpy` 深拷贝；`shared_ptr` 计数增减；`vector` push_back 扩容；`unique_ptr` 移动。
- [ ] C++17 的 `if constexpr` 与结构化绑定分别解决什么问题？

> ✅ 全部能回答 + 能默写代码，本课通过。